# 02 - Every ticker in the flat files

Scans the raw Polygon flat files and writes the complete list of symbols they contain. Unlike
notebook 01 this list is **survivorship free**: it holds every symbol that ever printed a bar,
including companies that went bankrupt or were acquired.

The command-line equivalent is `scripts/extract_tickers_from_flatfiles.py`.

Two details decide whether the result is correct, and the original version of this notebook got
both wrong. They are the subject of section 3.

## 1. Configuration

In [1]:
import json
import os
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from tqdm import tqdm     # plain tqdm: tqdm.auto wants ipywidgets and warns without it

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FLAT = Path(os.environ.get("POLYGON_FLATFILES", Path.home() / "local" / "flatfiles"))
SRC = FLAT / "day_aggs_v1"
OUT_DIR = REPO / "data" / "ticker_lists" / "refreshed"
N_WORKERS = max(4, (os.cpu_count() or 4) * 2)     # I/O bound, so oversubscribe

assert SRC.is_dir(), f"flat files not found at {SRC}  (set POLYGON_FLATFILES)"
files = sorted(SRC.rglob("*.csv.gz"))
print(f"source: {SRC}\n{len(files):,} daily flat files, {files[0].name} to {files[-1].name}, {N_WORKERS} threads")

source: /Users/mengren/local/flatfiles/day_aggs_v1
5,517 daily flat files, 2003-09-10.csv.gz to 2025-08-13.csv.gz, 32 threads


## 2. Scanning

A thread per file, each returning the set of symbols it saw. **Threads, not processes:** a
`ProcessPoolExecutor` cannot pickle a function defined in a notebook cell on any platform that
starts workers with *spawn*, which includes macOS and Windows, and fails with
`BrokenProcessPool`. The work here is gzip decompression and CSV parsing, both of which release
the GIL, so threads parallelise it nearly as well.

In [2]:
def tickers_from_file(csv_gz) -> set[str]:
    """Unique tickers in one .csv.gz, exactly as Polygon spells them.

    `keep_default_na=False` matters: pandas' default NA tokens include the string "NA", which is a
    real ticker (Nano Labs). Without this the symbol is read as null and silently dropped.
    The case is left alone; section 3 explains why.
    """
    out: set[str] = set()
    for chunk in pd.read_csv(csv_gz, usecols=["ticker"], dtype={"ticker": "string"},
                             compression="gzip", keep_default_na=False, na_values=[""],
                             chunksize=2_000_000):
        out.update(chunk["ticker"].dropna().unique())
    return out

all_tickers: set[str] = set()
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    for s in tqdm(pool.map(tickers_from_file, files), total=len(files), desc="scanning",
                  mininterval=10, miniters=len(files) // 4):   # a few lines of output, not hundreds
        all_tickers |= s

tickers = sorted(t for t in all_tickers if isinstance(t, str) and t.strip())
print(f"\nunique tickers: {len(tickers):,}")
print("NA (Nano Labs) present:", "NA" in tickers)

scanning:   0%|          | 0/5517 [00:00<?, ?it/s]

scanning:  40%|███▉      | 2197/5517 [00:10<00:15, 219.39it/s]

scanning:  75%|███████▍  | 4118/5517 [00:20<00:06, 202.87it/s]

scanning: 100%|██████████| 5517/5517 [00:29<00:00, 190.17it/s]


unique tickers: 33,833
NA (Nano Labs) present: True


## 3. The two mistakes this notebook used to make

### `keep_default_na`: the ticker `NA` disappears

pandas treats the literal string `NA` as a missing value by default. The earlier version of this
notebook printed a warning about "1 non-string ticker" and dropped it; that row was **Nano Labs Ltd**
(Nasdaq: `NA`), a real company with 773 day bars since its July 2022 IPO. The committed list carried
the damage until it was regenerated, so the cell below is now a regression check rather than a
finding: the committed list and a fresh scan have to agree, symbol for symbol.

In [3]:
committed_path = REPO / "data" / "ticker_lists" / "all_polygonio_tickers.json"
committed = json.loads(committed_path.read_text()) if committed_path.exists() else []
print(f"committed list: {len(committed):,} tickers, contains 'NA': {'NA' in committed}")
print(f"this scan     : {len(tickers):,} tickers, contains 'NA': {'NA' in tickers}")
pd.Series({"only in this scan": len(set(tickers) - set(committed)),
           "only in the committed list": len(set(committed) - set(tickers))}).to_frame("tickers")

committed list: 33,833 tickers, contains 'NA': True
this scan     : 33,833 tickers, contains 'NA': True


,tickers
only in this scan,0
only in the committed list,0


### Letter case carries meaning: do not upper-case

Polygon encodes the share class in the case of the symbol. Upper-casing merges securities that
are genuinely different companies or different instruments.

In [4]:
mixed = sorted(t for t in tickers if t != t.upper())
collisions = {k: sorted(v) for k, v in
              ((k, [t for t in tickers if t.upper() == k]) for k, n in Counter(t.upper() for t in tickers).items() if n > 1)}
print(f"mixed-case symbols: {len(mixed):,} of {len(tickers):,}  e.g. {mixed[:8]}")
print(f"symbols that would collide if upper-cased: {len(collisions)}")
pd.DataFrame([{"upper-cased": k, "distinct symbols": ", ".join(v)} for k, v in sorted(collisions.items())][:12])

mixed-case symbols: 4,041 of 33,833  e.g. ['AAGpT', 'AAGpT.CL', 'AAICpB', 'AAICpC', 'AAMpA', 'AAMpB', 'AANw', 'AAPw']
symbols that would collide if upper-cased: 125


,upper-cased,distinct symbols
0,AAP,"AAP, AAp"
1,AAPB,"AAPB, AApB"
2,AAPW,"AAPW, AAPw"
3,ABCW,"ABCW, ABCw"
4,ACIW,"ACIW, ACIw"
5,ADSW,"ADSW, ADSw"
6,AFW,"AFW, AFw"
7,AIPC,"AIPC, AIpC"
8,ALPA,"ALPA, ALpA"
9,AROW,"AROW, AROw"


The suffix letters are a code: `p` marks a preferred series (`AAGpT`) and `r` a right. A `w` is
ambiguous — it marks warrants (`TMCWW`) *and* when-issued lines, and Polygon types the latter as
common stock, so `AANw` is Aaron's when-issued rather than a warrant. Most of those never collide with a common-stock symbol, but about a hundred do, and
for those the two histories are merged.

Naming them from the reference table makes the cost obvious:

In [5]:
LAKE = Path(os.environ.get("POLYGON_LAKE_ROOT", Path.home() / "local" / "parquet_lake"))
ref_path = LAKE / "refdata" / "_market" / "market_tickers.parquet"
if ref_path.exists() and collisions:
    ref = pd.read_parquet(ref_path, columns=["ticker", "name", "type", "active"])
    rows = []
    for k in sorted(collisions)[:6]:
        for _, r in ref[ref["ticker"].str.upper() == k].iterrows():
            rows.append({"upper-cased": k, "variants": ", ".join(collisions[k]),
                         "name": r["name"], "type": r["type"], "active": r["active"]})
    display(pd.DataFrame(rows))
else:
    print("reference table not available; set POLYGON_LAKE_ROOT to name the colliding securities")

,upper-cased,variants,name,type,active
0,AAP,"AAP, AAp",ADVANCE AUTO PARTS INC,CS,True
1,AAP,"AAP, AAp",Alcoa Inc. $3.75 Preferred Stock,PFD,False
2,AAPB,"AAPB, AApB",GraniteShares ETF Trust GraniteShares 2x Long ...,ETF,True
3,AAPB,"AAPB, AApB",Alcoa Inc.,SP,False
4,AAPW,"AAPW, AAPw",Roundhill AAPL WeeklyPay ETF,ETF,True
5,AAPW,"AAPW, AAPw",ADVANCE AUTO PARTS INC WI,NaN,False
6,ABCW,"ABCW, ABCw",ANCHOR BANCORP WISCONSIN INC COMMON STOCK,NaN,False
7,ABCW,"ABCW, ABCw",AMERISOURCE BERGEN CORP COM EX-DIST WI,NaN,False
8,ACIW,"ACIW, ACIw","ACI Worldwide, Inc.",CS,True
9,ACIW,"ACIW, ACIw","ARCH COAL, INC. COM W.I.",NaN,False


> **Fixed, and the lakes were rebuilt.** `polygon_ingest` used to upper-case tickers on ingest, so
> these collisions were present in the built lakes: about 90 symbols carried two securities' bars on
> the same day. Symbols are now stored exactly as Polygon spells them, in the lakes and in the
> reference tables. Notebook 04 section 6 quantifies what it cost and what is left.

## 4. Saving

Written next to the committed lists rather than over them, exactly as in notebook 01.

In [6]:
def save_tickers(tickers, name: str, out_dir: Path = OUT_DIR) -> None:
    """Save a ticker collection as TXT (one per line), CSV and JSON."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    ordered = sorted(set(tickers))
    (out / f"{name}.txt").write_text("\n".join(ordered))
    pd.Series(ordered).to_csv(out / f"{name}.csv", index=False, header=False)
    (out / f"{name}.json").write_text(json.dumps(ordered))
    print(f"saved {len(ordered):,} tickers -> {out.relative_to(REPO)}/{name}.{{txt,csv,json}}")

save_tickers(tickers, "all_polygonio_tickers")

saved 33,833 tickers -> data/ticker_lists/refreshed/all_polygonio_tickers.{txt,csv,json}


## 5. What this list is good for

It is the complete symbol space, which makes it the right input for a **whole-market lake**
(README Step 3 with `--layout market`), and the wrong input for a research universe: it contains
warrants, rights, preferred series, when-issued lines and exchange test symbols alongside common
stock. Filter it with the reference table, or skip it entirely and build the point-in-time
universe, which applies those filters for you.